# Kapitel 14

In [1]:
# Tokenizer, Parser und Pretty-Printer in Python als Blackbox
def tokenize(s:str)->list:
    tokens = []
    i = 0
    while i < len(s):
        c = s[i]
        if c.isspace():
            i += 1
        elif c.isdigit() or (c == '.' and i+1 < len(s) and s[i+1].isdigit()):
            num = c
            i +=1
            while i < len(s) and (s[i].isdigit() or s[i] == '.'):
                num += s[i]
                i +=1
            tokens.append(num)
        elif c.isalpha():
            name = c
            i +=1
            while i < len(s) and s[i].isalnum():
                name += s[i]
                i +=1
            tokens.append(name)
        elif c in '^()+-*/,':      # Komma hinzugefügt
            tokens.append(c)
            i += 1
        else:
            i += 1
    return tokens

def is_number(s:str)->bool:
    try:
        int(s)
        return True
    except ValueError:
        return False

def is_floatnumber(s:str)->bool:
    try:
        float(s)
        return True
    except ValueError:
        return False

def parse_expression(tokens:list, pos:int)->tuple:
    lhs, pos = parse_term(tokens, pos)
    while pos < len(tokens) and tokens[pos] in ('+', '-'):
        op = tokens[pos]; pos += 1
        rhs, pos = parse_term(tokens, pos)
        lhs = [op, lhs, rhs]
    return lhs, pos

def parse_term(tokens:list, pos:int)->tuple:
    lhs, pos = parse_power(tokens, pos)
    while pos < len(tokens) and tokens[pos] in ('*', '/'):
        op = tokens[pos]; pos += 1
        rhs, pos = parse_power(tokens, pos)
        lhs = [op, lhs, rhs]
    return lhs, pos

def parse_power(tokens:list, pos:int)->tuple:
    # Basis: Klammer, Zahl, Variable, Funktionsaufruf, unärer Operator
    lhs, pos = parse_factor(tokens, pos)
    # Rechtsassoziative Kette von Exponenten
    if pos < len(tokens) and tokens[pos] == '^':
        pos += 1
        rhs, pos = parse_power(tokens, pos)
        lhs = ['^', lhs, rhs]
    return lhs, pos

def parse_factor(tokens:list, pos:int)->tuple:
    # Klammerausdruck
    if tokens[pos] == '(':
        pos += 1
        expr, pos = parse_expression(tokens, pos)
        if pos >= len(tokens) or tokens[pos] != ')':
            raise ValueError('Erwartet )')
        return expr, pos+1

    # Unärer +/-
    elif tokens[pos] in ('+', '-'):
        op = tokens[pos]; pos += 1
        factor, pos = parse_factor(tokens, pos)
        return [op, factor], pos

    # Zahl/Float
    elif is_number(tokens[pos]):
        num = int(tokens[pos]); pos += 1
        return num, pos
    elif is_floatnumber(tokens[pos]):
        num = float(tokens[pos]); pos += 1
        return num, pos

    # Name / Funktionsaufruf mit mehreren Argumenten
    elif tokens[pos][0].isalpha():
        name = tokens[pos]
        pos += 1
        if pos < len(tokens) and tokens[pos] == '(':
            pos += 1
            args = []
            # Falls nicht sofort ')' folgt, erstes Argument parsen
            if pos < len(tokens) and tokens[pos] != ')':
                arg, pos = parse_expression(tokens, pos)
                args.append(arg)
                # Weitere Argumente durch Komma getrennt
                while pos < len(tokens) and tokens[pos] == ',':
                    pos += 1
                    arg, pos = parse_expression(tokens, pos)
                    args.append(arg)
            # schließende Klammer
            if pos >= len(tokens) or tokens[pos] != ')':
                raise ValueError('Erwartet )')
            pos += 1
            # AST: [Funktion, Arg1, Arg2, …]
            return [name] + args, pos
        else:
            # reine Variable
            return name, pos
    else:
        raise ValueError('Unerwartetes Token: ' + tokens[pos])

def parse(s:str)->list:
    tokens = tokenize(s)
    expr, pos = parse_expression(tokens, 0)
    if pos != len(tokens):
        print(["Parser problem expr,pos=",expr,pos])
        raise ValueError('Unerwartetes Token am Ende')
    return expr

def to_string0(expr):
    if isinstance(expr, list):
        if len(expr) == 2 and isinstance(expr[0], str) and expr[0] not in '+-*/':
            # Funktion
            return expr[0] + '(' + to_string(expr[1]) + ')'
        elif len(expr) == 2 and expr[0] in ('+', '-'):
            # Unärer Operator
            return expr[0] + to_string(expr[1])
        elif len(expr) == 3:
            # Binärer Operator
            return '(' + to_string(expr[1]) + expr[0] + to_string(expr[2]) + ')'
        else:
            raise ValueError('Ungültiger Ausdruck')
    else:
        # Zahl oder Variable
        return str(expr)


def precedence(op:str)->int:
    if op in ('+', '-'): return 1
    elif op in ('*', '/'): return 2
    elif op == '^': return 3
    else: return 4

def is_operator(op:str)->bool:
    return op in ('+', '-', '*', '/', '^')

def is_right_associative(op:str)->bool: 
    return op == '^'

def to_string(expr, parent_op=None, is_right_child=False)->str:
    if isinstance(expr, list):
        # Funktion oder unärer Operator
        if len(expr) >= 2 and isinstance(expr[0], str) and not is_operator(expr[0]):
                # Funktionsaufruf: f(x, y, ...)
            args_str = ','.join(to_string(arg, parent_op=expr[0]) for arg in expr[1:])
            return expr[0] + '(' + args_str + ')'        
        #if len(expr) == 2 and isinstance(expr[0], str) and not is_operator(expr[0]):
            # Funktionsaufruf: f(x)
            # Funktionsklammern sind bereits um den Ausdruck, daher hier keine zusätzlichen Klammern nötig.
         #   return expr[0] + '(' + to_string(expr[1], parent_op=expr[0]) + ')'
        elif len(expr) == 2 and expr[0] in ('+', '-'):
            # Unärer Operator: +x oder -x
            return expr[0] + to_string(expr[1], parent_op=expr[0])
        elif len(expr) == 3 and is_operator(expr[0]):
            # Binärer Operator
            op = expr[0]
            left = expr[1]
            right = expr[2]

            current_prec = precedence(op)
            parent_prec = precedence(parent_op) if parent_op else 0
            
            left_str = to_string(left, parent_op=op, is_right_child=False)
            right_str = to_string(right, parent_op=op, is_right_child=True)

            # Standardmäßig keine Klammern:
            need_parens = False
            
            # Nur wenn der Parent ebenfalls ein Operator ist, prüfen wir auf Präzedenz
            # (Ist parent_op ein Funktionsname oder ein unärer Operator, setzen wir keine zusätzlichen Klammern)
            if parent_op is not None and is_operator(parent_op):
                if current_prec < parent_prec:
                    # Niedrigere Präzedenz -> Klammern
                    need_parens = True
                elif current_prec == parent_prec:
                    # Gleiche Präzedenz: bei rechtsassoziativen Operatoren '^' Klammern auf rechter Seite,
                    # bei '-' oder '/' auch auf rechter Seite sinnvoll, um Auswertungsreihenfolge klarzustellen.
                    if op in ('-', '/') and is_right_child:
                        need_parens = True
                    elif op == '^' and is_right_child:
                        need_parens = True

            expr_str = left_str + op + right_str
            return '(' + expr_str + ')' if need_parens else expr_str
        else:
            raise ValueError('Ungültiger Ausdruck: {}'.format(expr))
    else:
        # Zahl oder Variable
        return str(expr)
def testit(s:str):
    p=parse(s)
    s2=to_string(p)
    print(s," -> ",p," -> ",s2)
testit("2+3")
testit("x+3*y/2")
testit("pi*sin(x)^2")
testit("1/(x^2+y^2)")
testit("a+f(b)-c+d*9")

2+3  ->  ['+', 2, 3]  ->  2+3
x+3*y/2  ->  ['+', 'x', ['/', ['*', 3, 'y'], 2]]  ->  x+3*y/2
pi*sin(x)^2  ->  ['*', 'pi', ['^', ['sin', 'x'], 2]]  ->  pi*sin(x)^2
1/(x^2+y^2)  ->  ['/', 1, ['+', ['^', 'x', 2], ['^', 'y', 2]]]  ->  1/(x^2+y^2)
a+f(b)-c+d*9  ->  ['+', ['-', ['+', 'a', ['f', 'b']], 'c'], ['*', 'd', 9]]  ->  a+f(b)-c+d*9


In [2]:
# Bruchrechnung, Hilfsfunktionen etc. 
from fractions import Fraction
def istVar(x): 
    return type(x)==str and len(x)>=1 and x[0].isalpha()
def istZahl(x): 
    return type(x)==int or type(x)==float or type(x)==Fraction
def istSumme(x):
    if not(type(x)==list): return False
    if not(len(x)==3): return False
    return x[0]=="+"
def istDifferenz(x):
    if not(type(x)==list): return False
    if not(len(x)==3): return False
    return x[0]=="-"
def istProdukt(x):
    if not(type(x)==list): return False
    if not(len(x)==3): return False
    return x[0]=="*"
def istQuotient(x):
    if not(type(x)==list): return False
    if not(len(x)==3): return False
    return x[0]=="/"
def istPotenz(x):
    if not(type(x)==list): return False
    if not(len(x)==3): return False
    return x[0]=="^"
def istFunktion(x):
    if not(type(x)==list): return False
    return len(x)==2
    
def addiere(a,b): 
    if a==0: return b
    if b==0: return a
    if a==b: return multipliziere(2,a)
    if istZahl(a) and istZahl(b): return a+b
    return ["+",a,b]
def subtrahiere(a,b): 
    if a==b: return 0
    if b==0: return a
    if istZahl(a) and istZahl(b): return a-b
    return ["-",a,b]
def multipliziere(a,b):
    if a==0 or b==0: return 0
    if a==1: return b
    if b==1: return a
    if istZahl(a) and istZahl(b): return a*b
    if istPotenz(a) and istPotenz(b):
        if a[1]==b[1]: return potenziere(a[1],addiere(a[2],b[2]))
        return ["*",a,b]
    if istPotenz(a):
        if b==a[1]: return potenziere(a[1],addiere(1,a[2]))
        return ["*",a,b]
    if istPotenz(b):
        if a==b[1]: return potenziere(b[1],addiere(1,b[2]))
        return ["*",a,b]
    return ["*",a,b]
def dividiere(a,b): 
    if a==0: return 0
    if b==1: return a
    if a==b: return 1
    if type(a)==int and type(b)==int: return Fraction(a,b)
    if istZahl(a) and istZahl(b): return a/b
    return ["/",a,b]
def potenziere(a,b):
    if b==1: return a
    if b==0: return 1
    if a==0: return 0 
    if istPotenz(a): return potenziere(a[1],multipliziere(a[2],b))
    if istZahl(a) and istZahl(b): return a**b
    return ["^",a,b]
print("Ist 'a1' eine Variable?",istVar('a1'))
t=["+",4,"x"]
print("Ist ",t," eine Summe?",istSumme(t),", eine Differenz?",istDifferenz(t))
print("Addiere 5 und ['/',1,2] : ",addiere(5,dividiere(1,2)))
print("Multipliziere 1 und 'x' : ",multipliziere(1,"x"))

Ist 'a1' eine Variable? True
Ist  ['+', 4, 'x']  eine Summe? True , eine Differenz? False
Addiere 5 und ['/',1,2] :  11/2
Multipliziere 1 und 'x' :  x


In [3]:
# Substitution und Test auf Freiheit
def subst(term,alt,neu):
    if term==alt: return neu
    if type(term)==list:
        return [subst(x,alt,neu) for x in term]
    return term
def frei(term,var):
    if term==var: return False
    if type(term)==list:
        for x in term:
            if not(frei(x,var)): return False
    return True
print(frei(parse("a+1/x"),"x"))
print(subst(parse("(a+b)^2"),"a",parse("x+y")))

False
['^', ['+', ['+', 'x', 'y'], 'b'], 2]


In [4]:
def ableitung(term,var):
    if term==var: return 1
    if frei(term,var): return 0
    if istSumme(term): return addiere(ableitung(term[1],var),ableitung(term[2],var))
    if istDifferenz(term): return subtrahiere(ableitung(term[1],var),ableitung(term[2],var))
    if istProdukt(term): 
        u=term[1]; v=term[2]; us=ableitung(u,var); vs=ableitung(v,var)
        return addiere(multipliziere(us,v),multipliziere(u,vs))
    if istQuotient(term): 
        u=term[1]; v=term[2]; us=ableitung(u,var); vs=ableitung(v,var)
        return dividiere(subtrahiere(multipliziere(us,v),multipliziere(u,vs)),  potenziere(v,2))
    if istPotenz(term):
        a=term[1]; b=term[2]
        if frei(b,var): return multipliziere(multipliziere(b,potenziere(a,subtrahiere(b,1))),ableitung(a,var))
        return ableitung(["exp",["*",b,["ln",a]]],var)
    if istFunktion(term) and term[0]=="exp": return multipliziere(term,ableitung(term[1],var))
    if istFunktion(term) and term[0]=="ln": return dividiere(ableitung(term[1],var),term[1])
    if istFunktion(term) and term[0]=="sin": return multipliziere(["cos",term[1]],ableitung(term[1],var))
    if istFunktion(term) and term[0]=="cos": 
        return multipliziere(-1, multipliziere(["sin",term[1]],ableitung(term[1],var)))
    if istFunktion(term) and term[0]=="tan": 
        return multipliziere(addiere(1,potenziere(term[1],2)),ableitung(term[1],var))
    return ["Ableitung",term,var]
def strdiff(term:str,var:str)->str:
    return to_string(ableitung(parse(term),var))
t="3*x^2+b*x"; print(t,"' =",strdiff(t,"x"))
t="x^2*sin(a*x)"; print(t,"' =",strdiff(t,"x"))

3*x^2+b*x ' = 3*2*x+b
x^2*sin(a*x) ' = 2*x*sin(a*x)+x^2*cos(a*x)*a


In [5]:
import math
def vereinfache(term): # durchläuft einen Term und berechnet die Teilterme, die nur aus Zahlen bestehen
    if istVar(term) or istZahl(term): # Eine Variable/Zahl wird unverändert zurück geben 
        return term
    if istSumme(term): 
        a=vereinfache(term[1]); b=vereinfache(term[2])
        return addiere(a,b)
    if istDifferenz(term): 
        a=vereinfache(term[1]); b=vereinfache(term[2])
        return subtrahiere(a,b)
    if istProdukt(term): 
        a=vereinfache(term[1]); b=vereinfache(term[2])
        return multipliziere(a,b)
    if istQuotient(term): 
        a=vereinfache(term[1]); b=vereinfache(term[2])
        return dividiere(a,b)
    if istPotenz(term): 
        a=vereinfache(term[1]); b=vereinfache(term[2])
        return potenziere(a,b)
    if istFunktion(term):
        a=vereinfache(term[1])
        f=term[0]
        if f=="sqrt" and type(a)==int and int(math.sqrt(a))**2==a: 
            return int(math.sqrt(a))
        if f=="sin" and a==0: return 0
        if f=="sin" and a=="pi": return 0
        return [f,a]
    return term
to_string(vereinfache(parse("3+4+x+8/4*2+sin(a-a)+sqrt(4)")))

'7+x+4+2'

In [6]:
def alsPoly(term,var): # Ergebnis ist Liste [a_0,a_1,...] von a_0+a_1*x+a_2*x^2... 
    def Pmul(a:list,b:list)->list:
        res=[0]*(1+(len(a)-1)+(len(b)-1)) 
        # Ergebnis wird Polynom vom Grad(a)+Grad(b)
        for i in range(len(a)):
            for j in range(len(b)):
                res[i+j]=addiere(res[i+j],multipliziere(a[i],b[j]))
        return res    
    # oder [] falls es kein Polynom ist
    if term==var: return [0,1]
    if istZahl(term) or istVar(term): return [term]
    if istQuotient(term) or istFunktion(term): 
        raise Exception("Kein Polynom")
    if istPotenz(term):
        if type(term[2])==int and term[2]>=0:
            p=[1]
            a=alsPoly(term[1],var)
            if a==[]: return []
            for i in range(term[2]): p=Pmul(p,a)
            return p
        return []
    if istSumme(term):
        a=alsPoly(term[1],var)
        b=alsPoly(term[2],var)
        if len(a)<len(b): 
            a=a+[0]*(len(b)-len(a)) # mit Nullen auffüllen
        if len(a)>len(b): 
            b=b+[0]*(len(a)-len(b))
        return [addiere(a[i],b[i]) for i in range(len(a))]
    if istDifferenz(term):
        a=alsPoly(term[1],var)
        b=alsPoly(term[2],var)
        if a==[] or b==[]: return []
        if len(a)<len(b): 
            a=a+[0]*(len(b)-len(a)) # mit Nullen auffüllen
        if len(a)>len(b): 
            b=b+[0]*(len(a)-len(b))
        return [subtrahiere(a[i],b[i]) for i in range(len(a))]
    if istProdukt(term):
        a=alsPoly(term[1],var)
        b=alsPoly(term[2],var)
        if a==[] or b==[]: return []
        return Pmul(a,b)
    return []
print(alsPoly(parse("(x+3)^2"),"x"))
print(alsPoly(parse("3*x^2+b*x+5*x+9*(x-1)^2"),"x"))
def poly2term(L,var):
    res=0
    for k in range(len(L)):
        res=addiere(res,multipliziere(L[k],potenziere(var,k)))
    return res
print(poly2term([5,2,0,7],"x"))
print(to_string(poly2term(alsPoly(parse("3*x^2+b*x+5*x+9*(x-1)^2"),"x"),"x")))

[9, 6, 1]
[9, ['+', ['+', 'b', 5], -18], 12]
['+', ['+', 5, ['*', 2, 'x']], ['*', 7, ['^', 'x', 3]]]
9+(b+5+-18)*x+12*x^2


In [7]:
from sympy import *
x, y = symbols('x y')
print(x,type(x))
expr = (x + 2)*(x - 3)
print(expr)       
print(expr.func)   # Operator
print(expr.args)   # Operanden als Tupel

x <class 'sympy.core.symbol.Symbol'>
(x - 3)*(x + 2)
<class 'sympy.core.mul.Mul'>
(x - 3, x + 2)


In [8]:
simplify((x**2 - 1)/(x - 1))

x + 1

In [9]:
factor(x**2 - 5*x + 6) 

(x - 3)*(x - 2)

In [10]:
expand(expr)

x**2 - x - 6

In [11]:
expr.subs(x,y+1)

(y - 2)*(y + 3)

In [12]:
solve(x**2 - y*x + 6, x)  

[y/2 - sqrt(y**2 - 24)/2, y/2 + sqrt(y**2 - 24)/2]

In [13]:
solve(Eq(x+2, 5), x)

[3]

In [14]:
diff(x**3 + 2*x, x)    

3*x**2 + 2

In [15]:
integrate(3*x**2 + 2, x) 

x**3 + 2*x